In [0]:
%run /Workspace/weather_notebook/nb_utils_dev

In [0]:
print("="*60)
print("  AZURE ML TRAINING — UK WEATHER TEMPERATURE PREDICTION")
print("="*60)

In [0]:

# ── Load Gold ML features ─────────────────────────────────────
df_features = spark.read.format("delta").load(gold + "ml_features")

total_rows = df_features.count()
cities     = df_features.select("city_name").distinct().count()

print(f"\n  Total feature rows: {total_rows:,}")
print(f"  Cities:             {cities}")
print(f"\n  Schema:")
df_features.printSchema()

print(f"\n  Sample data:")
df_features.select(
    "city_name", "reading_timestamp",
    "temperature_c", "temp_lag_1h", "temp_lag_3h",
    "pressure_trend_3h", "temp_target_24h"
).show(5, truncate=False)

print(f"\n  Rows with target variable (temp_target_24h):")
with_target = df_features.filter(F.col("temp_target_24h").isNotNull()).count()
print(f"  {with_target:,} rows have 24h target")

In [0]:
# Install required packages
%pip install scikit-learn mlflow azureml-mlflow azureml-core --quiet
print("Packages installed")

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings("ignore")

print("[1/5] Preparing training data...")

# Convert to Pandas for sklearn
df = df_features.toPandas()

print(f"  Total rows: {len(df):,}")

# ── Encode categorical features ───────────────────────────────
le_city     = LabelEncoder()

df["city_encoded"]     = le_city.fit_transform(df["city_name"])

# ── Define feature columns ────────────────────────────────────
feature_columns = [
    "city_encoded",
    "latitude",
    "longitude",
    "reading_hour",
    "temperature_c",
    "humidity_pct",
    "pressure_hpa",
    "wind_speed_ms",
    "cloud_cover_pct",
    "is_daytime",
    "wind_beaufort",
]

# Add lag features if available
lag_features = [
    "temp_lag_1h", "temp_lag_3h",
    "temp_lag_6h", "temp_lag_24h",
    "humidity_lag_1h", "pressure_lag_1h",
    "wind_lag_1h", "pressure_trend_3h",
    "temp_trend_3h"
]

available_lags = [f for f in lag_features if f in df.columns]
feature_columns.extend(available_lags)

print(f"  Features: {len(feature_columns)}")
print(f"  Lag features available: {available_lags}")

# ── Filter rows with target ───────────────────────────────────
df_model = df[df["temp_target_24h"].notna()].copy()
df_model = df_model.fillna(0)

X = df_model[feature_columns]
y = df_model["temp_target_24h"]

print(f"  Training samples: {len(X):,}")
print(f"  Target range: {y.min():.1f}°C to {y.max():.1f}°C")

if len(X) < 10:
    print("\n  WARNING: Very few training samples.")
    print("  Model will train but accuracy may be low.")
    print("  More data accumulates every 20 minutes.")

In [0]:
print("\n[2/5] Splitting data...")

# Time-based split — last 20% as test
split_idx = int(len(df_model) * 0.8)

if split_idx < 5:
    # Not enough data — use random split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print("  Using random split (limited data)")
else:
    # Time-based split
    X_train = X.iloc[:split_idx]
    X_test  = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test  = y.iloc[split_idx:]
    print("  Using time-based split")

print(f"  Train: {len(X_train):,} rows")
print(f"  Test:  {len(X_test):,} rows")

In [0]:
for model_config in models_to_train:
    model_name = model_config["name"]
    model      = model_config["model"]
    params     = model_config["params"]

    print(f"\n  Training: {model_name}")

    with mlflow.start_run(run_name=model_name):

        # Log parameters
        mlflow.log_params(params)
        mlflow.log_param("features",      len(feature_columns))
        mlflow.log_param("train_samples", len(X_train))
        mlflow.log_param("test_samples",  len(X_test))
        mlflow.log_param("cities",        cities)

        # Train
        model.fit(X_train, y_train)

        # Evaluate
        y_pred = model.predict(X_test)
        mae    = mean_absolute_error(y_test, y_pred)
        rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
        r2     = r2_score(y_test, y_pred)

        # Log metrics
        mlflow.log_metric("mae_celsius",  round(mae, 4))
        mlflow.log_metric("rmse_celsius", round(rmse, 4))
        mlflow.log_metric("r2_score",     round(r2, 4))

        # Log model WITHOUT Unity Catalog registration
        mlflow.sklearn.log_model(
            model,
            artifact_path=model_name
            # removed: registered_model_name
        )

        results.append({
            "model":         model_name,
            "mae":           round(mae, 4),
            "rmse":          round(rmse, 4),
            "r2":            round(r2, 4),
            "trained_model": model
        })

        print(f"    MAE:  {mae:.2f}°C")
        print(f"    RMSE: {rmse:.2f}°C")
        print(f"    R²:   {r2:.4f}")

In [0]:
print("\n[4/5] Model comparison:")
print("="*50)
print(f"{'Model':<25} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-"*50)

best_model     = None
best_mae       = float("inf")
best_model_name = None

for r in results:
    print(f"{r['model']:<25} {r['mae']:>7.3f}°C {r['rmse']:>7.3f}°C {r['r2']:>8.4f}")
    if r["mae"] < best_mae:
        best_mae        = r["mae"]
        best_model      = r["trained_model"]
        best_model_name = r["model"]

print("="*50)
print(f"\n  Best model: {best_model_name}")
print(f"  Best MAE:   {best_mae:.2f}°C")
print(f"\n  Interpretation:")
print(f"  MAE = {best_mae:.2f}°C means on average the model")
print(f"  predicts temperature within {best_mae:.2f}°C of actual")

# Feature importance for tree models
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature":    feature_columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print(f"\n  Top 5 most important features:")
    for _, row in importance_df.head(5).iterrows():
        bar = "█" * int(row["importance"] * 50)
        print(f"  {row['feature']:<25} {bar} {row['importance']:.3f}")

In [0]:
print("\n[5/5] Saving best model and generating predictions...")

import pickle
import json

# ── Save model to ADLS Gen2 models container ──────────────────
model_adls_path = f"{models}temperature_predictor/"

# Save locally first then copy to ADLS
local_model_path = "/tmp/temperature_predictor_v1.pkl"
local_features_path = "/tmp/feature_names_v1.json"

with open(local_model_path, "wb") as f:
    pickle.dump(best_model, f)

with open(local_features_path, "w") as f:
    json.dump(feature_columns, f)

# Copy to ADLS Gen2 using dbutils
dbutils.fs.cp(f"file:{local_model_path}",
               f"{models}temperature_predictor/model_v1.pkl")
dbutils.fs.cp(f"file:{local_features_path}",
               f"{models}temperature_predictor/feature_names_v1.json")

print(f"  Model saved to: {models}temperature_predictor/")

# ── Generate predictions ──────────────────────────────────────
X_all    = df_model[feature_columns].fillna(0)
all_pred = best_model.predict(X_all)

predictions_df = pd.DataFrame({
    "city_name":          df_model["city_name"].values,
    "reading_timestamp":  df_model["reading_timestamp"].values,
    "reading_date":       df_model["reading_date"].values,
    "actual_temp_c":      df_model["temperature_c"].values,
    "predicted_temp_24h": all_pred,
    "actual_temp_24h":    df_model["temp_target_24h"].values,
    "error_celsius":      abs(df_model["temp_target_24h"].values - all_pred),
    "model_name":         best_model_name,
    "ingestion_date":     pd.Timestamp.now().date(),
    "ingestion_ts":       pd.Timestamp.now()
})

pred_spark = spark.createDataFrame(predictions_df)

pred_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold + "temperature_predictions")

total = pred_spark.count()
print(f"  Predictions saved: {total:,} rows")
print(f"  Location: {gold}temperature_predictions/")

print("\nSample predictions:")
pred_spark.select(
    "city_name", "actual_temp_c",
    "predicted_temp_24h", "error_celsius"
).show(5, truncate=False)

print("\n" + "="*60)
print("  ML TRAINING COMPLETE")
print(f"  Best model: {best_model_name}")
print(f"  MAE:        {best_mae:.2f}°C")
print(f"  Model:      {models}temperature_predictor/")
print(f"  Predictions:{gold}temperature_predictions/")
print("="*60)